# 타슈 수요예측 데이터 EDA

**목적**: 대전 공영자전거 타슈 이용데이터 + 환경데이터로 스테이션별·시간대별 대여/반납 수요예측 모델을 만들기 전에, 데이터의 구조와 품질을 파악한다.

| 테이블 | 내용 |
|---|---|
| `tassue_usage` | 타슈 대여/반납 이벤트 (타깃 데이터) |
| `tassue_station` | 대여소 마스터 |
| `air_quality_hourly` | 시간당 대기질 (에어코리아 성남동1 측정소) |
| `weather_hourly` | 시간당 날씨 |

> 추출 조건: 2025-01-01 이후 / `sun_times`는 2024년 데이터만 있어 제외
> 기존 `pm_hourly`는 2025-07부터 미세먼지가 끊겨서(이후 값은 weather와 중복), 에어코리아 확정자료로 새로 적재한 `air_quality_hourly`로 대체


## 0. 데이터 로드

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# csv는 eda 폴더의 상위 폴더(타슈관련)에 있음. 파일명 뒤 타임스탬프가 바뀌어도 되도록 glob으로 최신 파일을 잡는다
DATA_DIR = Path('..')
load = lambda prefix: pd.read_csv(sorted(DATA_DIR.glob(f'{prefix}_2*.csv'))[-1])

usage   = load('tassue_usage')
station = load('tassue_station')
aq      = load('air_quality_hourly')
weather = load('weather_hourly')

# 날짜/시간 파생 컬럼 미리 생성
usage['event_time'] = pd.to_datetime(usage['event_time'])
usage['date'] = usage['event_time'].dt.normalize()
usage['hour'] = usage['event_time'].dt.hour
usage['dow']  = usage['event_time'].dt.dayofweek   # 0=월 ~ 6=일

aq['ts_date']      = pd.to_datetime(aq['ts_date'])
weather['ts_date'] = pd.to_datetime(weather['ts_date'])

# 테이블 개요
pd.DataFrame([
    {'테이블': 'tassue_usage',   '행수': len(usage),   '컬럼수': usage.shape[1],   '기간': f"{usage['date'].min().date()} ~ {usage['date'].max().date()}"},
    {'테이블': 'tassue_station', '행수': len(station), '컬럼수': station.shape[1], '기간': '-'},
    {'테이블': 'air_quality_hourly', '행수': len(aq), '컬럼수': aq.shape[1], '기간': f"{aq['ts_date'].min().date()} ~ {aq['ts_date'].max().date()}"},
    {'테이블': 'weather_hourly', '행수': len(weather), '컬럼수': weather.shape[1], '기간': f"{weather['ts_date'].min().date()} ~ {weather['ts_date'].max().date()}"},
])

,테이블,행수,컬럼수,기간
0,tassue_usage,211035,8,2025-01-01 ~ 2026-04-01
1,tassue_station,19,3,-
2,air_quality_hourly,10176,9,2025-01-01 ~ 2026-03-01
3,weather_hourly,21384,7,2025-01-01 ~ 2026-06-24


---
## 1. tassue_usage — 이용데이터 (타깃)

우리가 예측하려는 값(대여/반납 건수)이 여기서 나온다. 구조부터 확인.

In [2]:
usage.head(10)

,id,event_type,event_time,station_id,qty,date,hour,dow
0,1,반납,2025-01-01 05:10:00,ST0685,2,2025-01-01,5,2
1,2,대여,2025-01-01 05:15:00,ST0639,2,2025-01-01,5,2
2,3,반납,2025-01-01 05:25:00,ST0065,2,2025-01-01,5,2
3,4,반납,2025-01-01 05:25:00,ST0627,2,2025-01-01,5,2
4,5,대여,2025-01-01 05:35:00,ST1302,2,2025-01-01,5,2
5,6,대여,2025-01-01 05:55:00,ST0627,2,2025-01-01,5,2
6,7,대여,2025-01-01 07:05:00,ST0685,2,2025-01-01,7,2
7,8,반납,2025-01-01 07:25:00,ST0685,2,2025-01-01,7,2
8,9,대여,2025-01-01 07:40:00,ST0720,2,2025-01-01,7,2
9,10,반납,2025-01-01 07:50:00,ST0065,2,2025-01-01,7,2


### 1-1. 기본 품질 체크

결측, 중복, 컬럼별 고유값을 한 번에 확인한다.

In [3]:
orig_cols = ['id','event_type','event_time','station_id','qty']  # 원본 컬럼만 (date/hour/dow는 우리가 만든 파생)
pd.DataFrame({
    '타입': usage[orig_cols].dtypes.astype(str),
    '결측수': usage[orig_cols].isna().sum(),
    '고유값수': usage[orig_cols].nunique(),
}).assign(비고=['PK', '대여/반납 2종', '5분 단위 스냅', '대여소 ID', '이벤트 묶음 건수'])

,타입,결측수,고유값수,비고
id,int64,0,211035,PK
event_type,object,0,2,대여/반납 2종
event_time,datetime64[ns],0,81639,5분 단위 스냅
station_id,object,0,19,대여소 ID
qty,int64,0,11,이벤트 묶음 건수


In [4]:
# (event_type, event_time, station_id) 조합이 중복되는 행이 있는지 → 0이어야 정상
print('중복행:', usage.duplicated(subset=['event_type','event_time','station_id']).sum())
usage['event_type'].value_counts().to_frame('건수').assign(비율=lambda d: (d['건수']/len(usage)*100).round(1))

중복행: 0


,건수,비율
event_type,,
반납,107297,50.8
대여,103738,49.2


**해석**: 결측·중복 없음. 대여/반납이 거의 반반 — 시스템 특성상 정상.

### 1-2. qty 컬럼의 의미 — 중요!

`event_time`이 전부 5분 단위(분이 5의 배수, 초는 0)로 찍혀있다. 즉 원천 데이터가 **5분 단위로 묶여서 수집**된 것이고, `qty`는 그 5분 동안 해당 스테이션에서 발생한 건수다.

→ **건수를 집계할 때는 행 개수(count)가 아니라 `qty`의 합(sum)을 써야 한다.**

In [5]:
print('분이 5의 배수가 아닌 행:', (usage['event_time'].dt.minute % 5 != 0).sum())
print('초가 0이 아닌 행:', (usage['event_time'].dt.second != 0).sum())
usage['qty'].value_counts().sort_index().to_frame('행수').assign(비율=lambda d: (d['행수']/len(usage)*100).round(2))

분이 5의 배수가 아닌 행: 0
초가 0이 아닌 행: 0


,행수,비율
qty,,
1,169422,80.28
2,34286,16.25
3,4848,2.30
4,1923,0.91
5,321,0.15
6,174,0.08
7,24,0.01
8,24,0.01
9,3,0.00


### 1-3. 시간대별 이용 패턴

시간대별 총 이용량(qty 합). 출퇴근 피크가 보이면 정상적인 공유자전거 패턴이다.

In [6]:
by_hour = usage.groupby('hour')['qty'].sum().to_frame('이용건수')
by_hour.style.bar(color='#5b9bd5')

,이용건수
hour,
0,4357
1,2859
2,1906
3,1307
4,1182
5,3850
6,6337
7,10530
8,14457


**해석**: 8시 출근 피크 + 15~18시(퇴근·하교) 최대 피크, 새벽 3~4시 최저. 전형적인 공유자전거 수요 곡선 → `hour`는 핵심 피처.

### 1-4. 요일 × 시간 패턴

요일(0=월 ~ 6=일)과 시간의 조합. 색이 진할수록 이용이 많다.

In [7]:
pivot_dh = usage.pivot_table(index='hour', columns='dow', values='qty', aggfunc='sum')
pivot_dh.columns = ['월','화','수','목','금','토','일']
pivot_dh.style.background_gradient(cmap='Blues', axis=None).format('{:.0f}')

,월,화,수,목,금,토,일
hour,,,,,,,
0,478,565,578,639,707,697,693
1,227,353,418,449,485,552,375
2,182,228,241,268,257,365,365
3,113,124,164,190,164,268,284
4,153,164,136,140,152,221,216
5,626,547,585,577,566,511,438
6,1040,933,1005,1016,1080,667,596
7,1795,1665,1807,1747,1826,852,838
8,2452,2148,2615,2473,2523,1277,969


**해석**: 평일은 8시/17~18시 쌍봉(출퇴근), 주말은 오후에 완만한 단봉(여가) — 요일과 시간의 **상호작용**이 존재하므로 모델이 둘의 조합을 학습할 수 있어야 한다 (트리 계열 모델이면 자동으로 처리됨).

### 1-5. 일별 이용량 추이와 결측일

일별로 집계해서 수집이 끊긴 구간을 찾는다.

In [8]:
daily = usage.groupby('date')['qty'].sum()
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
missing_days = full_range.difference(daily.index)

print(f'기간: {daily.index.min().date()} ~ {daily.index.max().date()}')
print(f'있어야 할 일수 {len(full_range)} / 실제 {len(daily)} / 빠진 날 {len(missing_days)}')
print(f'일평균 이용: {daily.mean():.0f}건')

# 빠진 날을 월별로 요약
pd.Series(missing_days).dt.to_period('M').value_counts().sort_index().to_frame('빠진 일수')

기간: 2025-01-01 ~ 2026-04-01
있어야 할 일수 456 / 실제 420 / 빠진 날 36
일평균 이용: 627건


,빠진 일수
2025-02,2
2025-03,3
2025-12,31


In [9]:
# 월별 이용량 — 계절성 + 12월 통째 누락 확인
monthly = usage.groupby(usage['date'].dt.to_period('M'))['qty'].sum().to_frame('이용건수')
monthly.style.bar(color='#70ad47')

,이용건수
date,
2025-01,19936
2025-02,8667
2025-03,14632
2025-04,22693
2025-05,24162
2025-06,24488
2025-07,19418
2025-08,20705
2025-09,22104


**발견사항 (중요)**
- **2025년 12월이 통째로 없음** (31일) + 2025-02-27 ~ 03-03 (5일) → 수집 누락으로 추정, DB에서 원인 확인 필요
- 마지막 날(2026-04-01)은 1건뿐 → 부분 수집이므로 학습에서 제외
- 결측일은 "수요가 0"이 아니라 "데이터가 없음"이므로 **0으로 채우면 안 되고 제외**해야 한다

### 1-6. 스테이션별 이용량

스테이션 간 수요 편차 확인. 편차가 크면 station별 특성 피처가 중요해진다.

In [10]:
by_st = usage.pivot_table(index='station_id', columns='event_type', values='qty', aggfunc='sum')
by_st = by_st.merge(station[['station_id','station_name']], left_index=True, right_on='station_id')
by_st = by_st[['station_id','station_name','대여','반납']].assign(합계=lambda d: d['대여']+d['반납']).sort_values('합계', ascending=False).reset_index(drop=True)
by_st.style.bar(subset=['합계'], color='#ffc000')

,station_id,station_name,대여,반납,합계
0,ST0065,대전역_대한통운 건너편,26568,26618,53186
1,ST0685,대흥동 이안과병원 정류장,21584,21976,43560
2,ST1067,중앙로역 3번출구,20612,20638,41250
3,ST0627,대전역 동광장 버스정류장,9605,9623,19228
4,ST0107,중앙시장 공영주차장,8397,8410,16807
5,ST0056,이스트시티_103동 화병원 건너편,7289,7253,14542
6,ST0641,원동 중앙프라자 정류장,6925,6941,13866
7,ST0720,대흥교 원동네거리 방면,4880,4878,9758
8,ST0639,원동네거리 요양병원,3827,3838,7665
9,ST0642,정동 하나은행 건너편,3502,3503,7005


---
## 2. tassue_station — 대여소 마스터

usage에 나오는 station_id가 마스터에 다 있는지(참조 무결성) 확인한다.

In [11]:
u_ids, m_ids = set(usage['station_id']), set(station['station_id'])
print(f'usage 스테이션 {len(u_ids)}개 / 마스터 {len(m_ids)}개')
print('마스터에만 있는 ID:', m_ids - u_ids or '없음')
print('usage에만 있는 ID (마스터 누락):', u_ids - m_ids or '없음')
station.sort_values('station_id').reset_index(drop=True)

usage 스테이션 19개 / 마스터 19개
마스터에만 있는 ID: 없음
usage에만 있는 ID (마스터 누락): 없음


,id,station_id,station_name
0,20,ST0056,이스트시티_103동 화병원 건너편
1,21,ST0065,대전역_대한통운 건너편
2,22,ST0082,대전역 동광장 휠타이어
3,23,ST0107,중앙시장 공영주차장
4,24,ST0342,신안동 블루부코
5,25,ST0627,대전역 동광장 버스정류장
6,26,ST0628,신안동 대전화병원_베스티안 화상병원
7,27,ST0639,원동네거리 요양병원
8,28,ST0641,원동 중앙프라자 정류장
9,29,ST0642,정동 하나은행 건너편


**해석**: 19개 스테이션이 양쪽에 100% 매칭 → 조인 시 유실 없음.

---
## 3. air_quality_hourly — 시간당 대기질 (미세먼지)

기존 pm_hourly는 2025-07부터 미세먼지 수집이 끊겨서, **에어코리아 성남동1(도시대기) 확정자료**를 받아 새로 적재한 테이블이다. 월별로 받은 시간자료를 합쳤고, 24시 표기는 다음날 0시로 변환했다.

In [12]:
aq.head()

,id,ts_date,hour,pm10,pm25,ozone,no2,co,so2
0,1,2025-01-01,1,20.0,13.0,0.0058,0.0362,0.59,0.0042
1,2,2025-01-01,2,23.0,17.0,0.0046,0.0375,0.82,0.0047
2,3,2025-01-01,3,22.0,13.0,0.0030,0.0373,0.73,0.0048
3,4,2025-01-01,4,24.0,7.0,0.0063,0.0315,0.64,0.0045
4,5,2025-01-01,5,14.0,7.0,0.0108,0.0254,0.52,0.0039


### 3-1. 결측 확인

컬럼별 결측률과, 월별로 커버가 꽉 차 있는지 확인한다.

In [13]:
(aq.isna().mean()*100).round(1).to_frame('결측률(%)')

,결측률(%)
id,0.0
ts_date,0.0
hour,0.0
pm10,1.3
pm25,3.5
ozone,2.1
no2,1.8
co,1.6
so2,1.6


In [14]:
aq['month'] = aq['ts_date'].dt.to_period('M')
aq.groupby('month').agg(
    행수=('hour','count'),
    pm10결측=('pm10', lambda s: int(s.isna().sum())),
    pm25결측=('pm25', lambda s: int(s.isna().sum())),
    pm10평균=('pm10','mean'), pm25평균=('pm25','mean')).round(1)

,행수,pm10결측,pm25결측,pm10평균,pm25평균
month,,,,,
2025-01,743,4,21,36.8,19.0
2025-02,672,5,7,32.3,15.5
2025-03,744,3,15,47.0,21.1
2025-04,720,5,6,39.3,16.3
2025-05,744,3,4,31.3,15.6
2025-06,720,52,75,29.3,18.8
2025-07,744,4,34,19.8,11.6
2025-08,744,31,94,15.7,9.7
2025-09,720,11,57,13.9,7.7


**해석**: 2025-01 ~ 2026-02 전 구간이 꽉 차 있다 (2026-03 행 1개는 2/28 24시 → 3/1 0시 변환분). 결측은 pm10 1.3%, pm25 3.5%로 측정소 점검 시간대 수준 — 보간으로 처리 가능. 월별 평균을 보면 겨울~봄에 높고 여름~가을에 낮은 계절성이 뚜렷하다.

**주의**: 미세먼지는 2026-02까지만 있다 (에어코리아 확정자료가 아직 3월 미공개). 이용데이터는 2026-03까지 있으므로, 미세먼지를 피처로 쓰면 학습 구간이 한 달 짧아진다.

### 3-2. 값 분포

In [15]:
aq[['pm10','pm25','ozone','no2','co','so2']].describe().round(3)

,pm10,pm25,ozone,no2,co,so2
count,10039.000,9821.000,9965.000,9994.000,10018.000,10018.000
mean,30.912,15.272,0.035,0.017,0.396,0.003
std,23.197,12.692,0.020,0.013,0.135,0.001
min,1.000,1.000,0.001,0.002,0.160,0.002
25%,17.000,6.000,0.021,0.008,0.310,0.002
50%,26.000,12.000,0.035,0.012,0.360,0.003
75%,39.000,21.000,0.047,0.021,0.440,0.003
max,520.000,106.000,0.112,0.084,1.490,0.009


**해석**: pm10 최대 520은 황사 이벤트로 보이며 실제 값일 가능성이 높아 그대로 둔다. 나머지도 정상 범위.

---
## 4. weather_hourly — 시간당 날씨

메인 환경 피처. 결측과 값 범위를 확인한다.

In [16]:
weather.head()

,id,ts_date,hour,temp,humid,wind,precip
0,38289,2025-01-01,0,-2.19,59.33,0.91,0.0
1,38290,2025-01-01,1,-2.56,62.22,0.74,0.0
2,38291,2025-01-01,2,-2.80,64.76,0.69,0.0
3,38292,2025-01-01,3,-2.88,66.23,0.62,0.0
4,38293,2025-01-01,4,-3.28,69.10,0.82,0.0


In [17]:
print('컬럼별 결측:', weather.isna().sum().sum(), '개')

# (날짜, 시간) 단위로 빠진 시간대가 있는지
full_idx = pd.MultiIndex.from_product([pd.date_range(weather['ts_date'].min(), weather['ts_date'].max(), freq='D'), range(24)], names=['ts_date','hour'])
have_idx = pd.MultiIndex.from_frame(weather[['ts_date','hour']])
missing_hours = full_idx.difference(have_idx)
print(f'빠진 시간대: {len(missing_hours)}개 / 전체 {len(full_idx)}개 ({len(missing_hours)/len(full_idx)*100:.1f}%)')

weather[['temp','humid','wind','precip']].describe().round(2)

컬럼별 결측: 8 개
빠진 시간대: 144개 / 전체 12960개 (1.1%)


,temp,humid,wind,precip
count,21383.00,21379.00,21383.00,21383.00
mean,13.71,64.57,1.80,0.42
std,11.02,21.48,1.18,3.15
min,-12.83,8.16,0.00,0.00
25%,4.50,49.00,0.98,0.00
50%,14.30,66.00,1.60,0.00
75%,22.90,82.90,2.50,0.00
max,36.50,99.00,8.12,78.30


**해석**: 값 결측 0%, 빠진 시간대도 극소수(보간으로 처리 가능). 온도 -12.8~36.5℃, 강수 최대 78mm — 이상치 없음. 품질이 가장 좋은 테이블.

### 4-1. 월별 평균 기온·강수 (계절성 확인)

In [18]:
wx_m = weather.groupby(weather['ts_date'].dt.to_period('M')).agg(평균기온=('temp','mean'), 총강수=('precip','sum')).round(1)
wx_m.style.background_gradient(subset=['평균기온'], cmap='coolwarm').bar(subset=['총강수'], color='#5b9bd5')

,평균기온,총강수
ts_date,,
2025-01,0.500000,213.900000
2025-02,-0.500000,155.900000
2025-03,8.600000,625.500000
2025-04,14.100000,652.100000
2025-05,17.900000,1032.100000
2025-06,24.000000,3368.200000
2025-07,28.200000,601.600000
2025-08,28.200000,211.000000
2025-09,23.400000,722.200000


---
## 5. 날짜 정합성 & 결론

### 테이블별 커버리지

In [19]:
pd.DataFrame([
    {'테이블':'tassue_usage',   '시작':'2025-01-01', '끝':str(usage['date'].max().date()),   '비고':'12월 통째 + 2월말 5일 누락, 마지막 날 부분수집'},
    {'테이블':'air_quality_hourly', '시작':'2025-01-01', '끝':'2026-02-28', '비고':'에어코리아 백필 완료, 2026-03 확정자료 미공개'},
    {'테이블':'weather_hourly', '시작':'2025-01-01', '끝':str(weather['ts_date'].max().date()), '비고':'결측 없음'},
])

,테이블,시작,끝,비고
0,tassue_usage,2025-01-01,2026-04-01,"12월 통째 + 2월말 5일 누락, 마지막 날 부분수집"
1,air_quality_hourly,2025-01-01,2026-02-28,"에어코리아 백필 완료, 2026-03 확정자료 미공개"
2,weather_hourly,2025-01-01,2026-06-24,결측 없음


### 결론 — 전처리 방향

1. **학습 구간**: 2025-01-01 ~ 2026-03-31, 이용데이터 결측일 36일 제외 → 약 **419일**
2. **타깃 테이블**: (station_id × date × hour) 조합을 전부 만들고 → usage를 `qty 합`으로 집계해 붙이고 → 없는 조합은 **0으로 채움** (단, 결측일은 조합 자체에서 제외)
3. **피처 조인**: weather_hourly와 air_quality_hourly를 (date, hour)로 조인. 미세먼지는 2026-02까지라 2026-03 구간은 제외하거나 결측 허용 모델 사용
4. **파생 피처**: hour, 요일, 주말/공휴일, 계절, lag(전주 동시간 수요) 등
5. 예상 학습 데이터: 419일 × 24시간 × 19스테이션 ≈ **19만 행**
